# CP2 Week 6 -- Report Generation

**Course:** Computer Programming 2 (CP2)
**Prerequisites:** Weeks 1-5
**Focus:** JSON reports, Markdown generation, interpretation text

## Learning Objectives
- Generate structured JSON reports
- Create readable Markdown reports
- Add interpretation text to analysis results
- Build a complete export_results() function

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: JSON Report Generation

JSON is the standard format for machine-readable reports. Your v2 pipeline should export a report.json with all results.

In [ ]:
import json, os
from datetime import datetime

def generate_json_report(clean_data, results, config):
    """Generate a complete JSON report.
    
    Returns:
        dict: the report (also saved to disk)
    """
    report = {
        "project_name": config.get("project_name", "unknown"),
        "track": config.get("track", "unknown"),
        "version": config.get("version", "v2"),
        "generated_at": datetime.now().isoformat(),
        "dataset": {
            "n_raw": config.get("n_raw", 0),
            "n_clean": len(clean_data),
            "n_dropped": config.get("n_raw", 0) - len(clean_data),
        },
        "cleaning_summary": results.get("cleaning_summary", {}),
        "analysis_summary": results.get("analysis_summary", {}),
        "figures": results.get("figures", []),
    }
    
    path = config.get("report_path", "reports/report.json")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(report, f, indent=2)
    
    print("Report saved: " + path)
    return report

# Demo
config = {
    "project_name": "demo",
    "track": "data",
    "version": "v2",
    "n_raw": 100,
    "report_path": "reports/report.json",
}
clean = [{"v": i} for i in range(80)]
results = {
    "analysis_summary": {"mean": 39.5, "std": 23.1, "count": 80},
    "cleaning_summary": {"missing": 10, "outlier": 10},
    "figures": ["timeseries.png", "summary.png"],
}

report = generate_json_report(clean, results, config)
print(json.dumps(report, indent=2))

**Expected Output:**
```
Report saved: reports/report.json
(JSON output with all fields)
```

---
## Part 2: Markdown Report Generation

Markdown is human-readable. Convert your JSON report to a nicely formatted Markdown document.

In [ ]:
def generate_markdown_report(report_dict):
    """Convert a JSON report to readable Markdown."""
    lines = []
    lines.append("# " + report_dict["project_name"] + " -- Analysis Report")
    lines.append("")
    lines.append("**Track:** " + report_dict["track"]
                 + " | **Version:** " + report_dict["version"])
    lines.append("**Generated:** " + report_dict.get("generated_at", "N/A"))
    lines.append("")
    
    ds = report_dict.get("dataset", {})
    lines.append("## Dataset Summary")
    lines.append("- Raw rows: " + str(ds.get("n_raw", 0)))
    lines.append("- Clean rows: " + str(ds.get("n_clean", 0)))
    lines.append("- Dropped: " + str(ds.get("n_dropped", 0)))
    lines.append("")
    
    cs = report_dict.get("cleaning_summary", {})
    if cs:
        lines.append("## Cleaning Summary")
        for reason, count in cs.items():
            lines.append("- " + reason + ": " + str(count))
        lines.append("")
    
    ans = report_dict.get("analysis_summary", {})
    if ans:
        lines.append("## Analysis Results")
        for metric, value in ans.items():
            lines.append("- " + metric + ": " + str(value))
        lines.append("")
    
    return "\n".join(lines)

md_text = generate_markdown_report(report)
print(md_text)

# Save it
with open("reports/report.md", "w") as f:
    f.write(md_text)
print("\nSaved: reports/report.md")

**Expected Output:**
```
# demo -- Analysis Report
**Track:** data | **Version:** v2
(rest of markdown output)

Saved: reports/report.md
```

---
## Part 3: Adding Interpretation Text

Numbers alone are not enough. Add human-readable interpretation.

In [ ]:
def interpret_results(summary):
    """Generate interpretation text from analysis summary."""
    lines = []
    
    mean = summary.get("mean", 0)
    std = summary.get("std", 0)
    count = summary.get("count", 0)
    
    lines.append("Based on " + str(count) + " clean records:")
    
    # Variability assessment
    cv = (std / mean * 100) if mean != 0 else 0
    if cv < 10:
        lines.append("- Data is very consistent (CV=" + str(round(cv, 1)) + "%)")
    elif cv < 25:
        lines.append("- Data has moderate variability (CV=" + str(round(cv, 1)) + "%)")
    else:
        lines.append("- Data is highly variable (CV=" + str(round(cv, 1)) + "%)")
    
    return "\n".join(lines)

summary = {"mean": 39.5, "std": 23.1, "count": 80}
print(interpret_results(summary))

**Expected Output:**
```
Based on 80 clean records:
- Data is highly variable (CV=58.5%)
```

---
## Part 4: Complete export_results() Function

This function ties everything together: it creates the JSON report, Markdown report, and interpretation text in one call.

In [ ]:
def export_results(clean_data, results, figures, config):
    """Export all results: JSON report, Markdown report, interpretation.
    
    Args:
        clean_data: cleaned data rows
        results: dict with analysis_summary, cleaning_summary
        figures: list of figure file paths
        config: pipeline configuration
    
    Returns:
        dict with paths to all exported files
    """
    import json, os
    from datetime import datetime
    
    # Build report
    report = {
        "project_name": config.get("project_name", "unknown"),
        "track": config.get("track", "unknown"),
        "version": config.get("version", "v2"),
        "generated_at": datetime.now().isoformat(),
        "dataset": {
            "n_raw": config.get("n_raw", 0),
            "n_clean": len(clean_data),
        },
        "cleaning_summary": results.get("cleaning_summary", {}),
        "analysis_summary": results.get("analysis_summary", {}),
        "figures": figures,
    }
    
    # Save JSON
    json_path = config.get("report_path", "reports/report.json")
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    with open(json_path, "w") as f:
        json.dump(report, f, indent=2)
    print("Saved JSON: " + json_path)
    
    # Save Markdown
    md_text = generate_markdown_report(report)
    md_path = json_path.replace(".json", ".md")
    with open(md_path, "w") as f:
        f.write(md_text)
    print("Saved Markdown: " + md_path)
    
    # Add interpretation
    interp = interpret_results(results.get("analysis_summary", {}))
    print("\nInterpretation:")
    print(interp)
    
    return {"json": json_path, "markdown": md_path}

# Demo
config = {"project_name": "demo", "track": "data", "version": "v2",
          "n_raw": 100, "report_path": "reports/report.json"}
clean = [{"v": i} for i in range(80)]
results = {
    "analysis_summary": {"mean": 39.5, "std": 23.1, "count": 80},
    "cleaning_summary": {"missing": 10, "outlier": 10},
}
exported = export_results(clean, results, ["timeseries.png"], config)

**Expected Output:**
```
Saved JSON: reports/report.json
Saved Markdown: reports/report.md

Interpretation:
Based on 80 clean records:
- Data is highly variable (CV=58.5%)
```

---
## Part 5: Report Tables

Add formatted tables to your Markdown report for cleaner presentation.

In [ ]:
def format_as_table(headers, rows):
    """Format data as a Markdown table."""
    # Calculate column widths
    widths = [len(h) for h in headers]
    for row in rows:
        for i, val in enumerate(row):
            widths[i] = max(widths[i], len(str(val)))
    
    # Build table
    lines = []
    # Header row
    header = "| " + " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers)) + " |"
    lines.append(header)
    # Separator
    sep = "| " + " | ".join("-" * widths[i] for i in range(len(headers))) + " |"
    lines.append(sep)
    # Data rows
    for row in rows:
        line = "| " + " | ".join(str(v).ljust(widths[i]) for i, v in enumerate(row)) + " |"
        lines.append(line)
    
    return "\n".join(lines)

# Demo: cleaning summary table
table = format_as_table(
    ["Reason", "Count", "Percent"],
    [
        ["Missing values", "10", "10.0%"],
        ["Non-numeric", "5", "5.0%"],
        ["Out of range", "5", "5.0%"],
    ]
)
print(table)

**Expected Output:**
```
| Reason         | Count | Percent |
| -------------- | ----- | ------- |
| Missing values | 10    | 10.0%   |
| Non-numeric    | 5     | 5.0%    |
| Out of range   | 5     | 5.0%    |
```

### Try It Yourself

Create a table showing your analysis results (mean, std, min, max, count). Use format_as_table.

In [ ]:
# Try it: create an analysis results table
# TODO: use format_as_table to show your results


### Common Mistakes: Report Generation

**Mistake 1:** Not including timestamp -- you cannot tell when the report was created.

**Fix:** Always include datetime.now().isoformat() in your report.

**Mistake 2:** Forgetting os.makedirs -- report saving crashes if directory does not exist.

**Fix:** Always call os.makedirs(dir, exist_ok=True) before writing files.

**Mistake 3:** Putting raw data in the report -- reports should be summaries, not data dumps.

**Fix:** Reports contain summary statistics, not individual rows.


### Debugging Tips: Report Generation

- If json.dump fails, check that all values are JSON-serializable (no sets, no custom objects)
- If Markdown looks wrong, check for missing newlines between sections
- If datetime is wrong, make sure you import from datetime, not import datetime
- Use json.dumps(report, indent=2) to preview before saving

### Key Takeaway

- JSON reports are machine-readable, Markdown reports are human-readable
- Your pipeline should generate BOTH
- Add interpretation text that explains what the numbers mean
- Always include metadata: project name, track, version, timestamp
- Use tables in Markdown for cleaner data presentation
- export_results() is the single function that saves everything

---
## Mini-Quiz

In [ ]:
# Q1: What is the difference between JSON and Markdown reports?
# Answer: 

# Q2: What metadata should every report include?
# Answer: 

# Q3: Why add interpretation text to analysis results?
# Answer: 

# Q4: What does CV (coefficient of variation) tell you?
# Answer: 

---
## Homework: 12 Exercises

### Review (1-4)

In [ ]:
# HW1: Generate a JSON report for your project.


In [ ]:
# HW2: Generate a Markdown report from your JSON.


In [ ]:
# HW3: Add interpretation to your analysis results.


In [ ]:
# HW4: What metadata should every report include?


### Practice (5-8)

In [ ]:
# HW5: Add a "recommendations" section to your report.


In [ ]:
# HW6: Include figure file paths in your report.


In [ ]:
# HW7: Write a function that loads a report.json and prints a summary.


In [ ]:
# HW8: Add a "data lineage" section showing all cleaning steps.


### Challenge (9-11)

In [ ]:
# HW9: Generate an HTML report (simple version).


In [ ]:
# HW10: Add tables to your Markdown report.


In [ ]:
# HW11: Create a report comparison function (v1 vs v2 results).


### Mini-Project

In [ ]:
# HW12: Build a complete reporting module. JSON + Markdown + interpretation.


---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)